# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR\u02c6\u00b2 Mass Spectrometry dataset using the `mlcroissant` Python library.

### Dataset Source
The dataset source is provided as a Croissant schema at the URL below.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the FAIR\u02c6\u00b2 Mass Spectrometry dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Show basic dataset metadata
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values. These IDs are used for reference in all further operations.

In [ ]:
# List all record sets and their @id
print("Available record sets and their @id values:")
record_sets = [rs['@id'] for rs in dataset.metadata.record_sets]
for rs in dataset.metadata.record_sets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name','(no name)')}")

# Show fields (@id and name) in each record set
for rs in dataset.metadata.record_sets:
    print(f"\nFields for record set @id '{rs['@id']}' ({rs.get('name','(no name)')}):")
    for field in rs.get('fields', []):
        print(f"  - @id: {field['@id']}, name: {field.get('name','(no name)')}, dataType: {field.get('dataType', '')}")

## 3. Data Extraction
Load data from a selected record set into a DataFrame for analysis. Use the record set and field `@id`s you identified above.

In [ ]:
dataframes = {}

# For this dataset, the primary record set appears to be 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034' (as the only distribution in the schema)
main_recordset_id = "https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034"
# If dataset.metadata.record_sets is empty but files/distributions have 'record sets', adjust as needed

# Some datasets have one record set associated with the data file/distribution
try:
    records = list(dataset.records(record_set=main_recordset_id))
    df = pd.DataFrame(records)
    dataframes[main_recordset_id] = df
    print(f"Loaded dataframe columns: {df.columns.tolist()}")
    display(df.head())
except Exception as e:
    print(f"Could not load records for the main record set: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalization, and grouping.

We'll use the field `cr:abundance` (abundance), which is a typical quantitative protein field in mass spectrometry datasets. Adjust the field `@id` as needed after reviewing the real fields printed above.

In [ ]:
# Replace the following with the real field/column @id for 'abundance', e.g. 'cr:abundance'
numeric_field = None
# Show all columns to help user identify numeric fields
print("DataFrame columns:", dataframes[main_recordset_id].columns.tolist())
# Try to auto-detect an abundance or numeric field
for col in dataframes[main_recordset_id].columns:
    if 'abundance' in col.lower() or 'amount' in col.lower() or 'intensity' in col.lower():
        numeric_field = col
        break

if numeric_field is None:
    numeric_field = dataframes[main_recordset_id].select_dtypes('number').columns[0]
    print(f"Defaulting to first numeric field: {numeric_field}")
else:
    print(f"Using numeric field: {numeric_field}")

# Set a filtering threshold
threshold = dataframes[main_recordset_id][numeric_field].mean() if numeric_field else 0
filtered_df = dataframes[main_recordset_id][dataframes[main_recordset_id][numeric_field] > threshold]
print(f"Number of records with {numeric_field} > {threshold:.3f}: {filtered_df.shape[0]}")
display(filtered_df.head())

# Normalize the numeric field (z-score normalization)
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by an annotation/category. We'll pick the first non-numeric field.
non_numeric_cols = dataframes[main_recordset_id].select_dtypes(exclude='number').columns.tolist()
group_field = None
if non_numeric_cols:
    group_field = non_numeric_cols[0]

if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped statistics by '{group_field}':")
    display(grouped_df.head())

## 5. Visualization
Visualize the selected numeric field distribution and its relation to a grouping field (if found).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the normalized numeric field
plt.figure(figsize=(8,4))
sns.histplot(filtered_df[f"{numeric_field}_normalized"], bins=30, kde=True)
plt.title(f'Distribution of normalized {numeric_field}')
plt.xlabel(f'{numeric_field} (normalized)')
plt.ylabel('Count')
plt.show()

# If grouping field exists, show boxplot of numeric_field by group_field
if group_field:
    plt.figure(figsize=(10,4))
    sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
    plt.title(f'{numeric_field} by {group_field}')
    plt.xticks(rotation=60)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook we loaded and explored the FAIR\u02c6\u00b2 Mass Spectrometry dataset using `mlcroissant`.

- We loaded the dataset via its Croissant JSON-LD schema.
- We discovered record set and field `@id`s to facilitate careful referencing and reproducibility.
- We extracted the main data to a DataFrame, selected a key numeric field for further exploration, performed filtering and normalization, and visualized the results.
- We grouped data by a categorical field (where present), to investigate potential summary statistics.

This workflow demonstrates best practices for reproducible and efficient FAIR dataset exploration using `mlcroissant`.